<a href="https://colab.research.google.com/github/PraiseOrly/Medibot/blob/main/Medical_Question_Answering_Chatbot2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **PROJECT NAME:**

# MediBot: ChatDoctor QA

---
(MediBot is a Python-based medical chatbot that answers queries using the chatdoctor_icliniq dataset from Hugging Face. It employs fuzzy string matching to find relevant question-answer pairs from ChatDoctor and iCliniq sources, delivering accurate responses via a command-line interface.)

Install necessary libraries

In [17]:
!pip install datasets
!pip install fuzzywuzzy
!pip install python-Levenshtein

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
!cp "/content/drive/My Drive/Colab Notebooks/Medical_Question_Answering_Chatbot2.ipynb" /content/

In [20]:
import os
print("Files in /content:", os.listdir('/content'))

Files in /content: ['.config', 'drive', 'MediBot.ipynb', 'Medical_Question_Answering_Chatbot2.ipynb', 'sample_data']


In [21]:
!mv /content/Medical_Question_Answering_Chatbot2.ipynb /content/MediBot.ipynb

Clean Metadata


In [22]:
import json
import os

# Specify the notebook filename (update after verifying with os.listdir())
notebook_file = 'Medical_Question_Answering_Chatbot2.ipynb'  # Replace with your notebook’s name, e.g., 'Medical_Question_Answering_Chatbot.ipynb'

# Check if the file exists
if not os.path.exists(notebook_file):
    print(f"Error: File '{notebook_file}' not found in {os.getcwd()}")
    print("Files in directory:", os.listdir())
    print("Please save the notebook to /content or update 'notebook_file'.")
else:
    # Read the notebook
    with open(notebook_file, 'r') as f:
        nb = json.load(f)

    # Remove metadata.widgets if it exists
    if 'widgets' in nb.get('metadata', {}):
        del nb['metadata']['widgets']
        print("Removed metadata.widgets")
    else:
        print("No metadata.widgets found; notebook metadata is clean")

    # Save the modified notebook
    with open(notebook_file, 'w') as f:
        json.dump(nb, f, indent=2)
        print(f"Saved cleaned notebook to {notebook_file}")

Error: File 'Medical_Question_Answering_Chatbot2.ipynb' not found in /content
Files in directory: ['.config', 'drive', 'MediBot.ipynb', 'sample_data']
Please save the notebook to /content or update 'notebook_file'.


Import necessary libraries

In [23]:
from datasets import load_dataset
from fuzzywuzzy import fuzz
import sys
import random

Load Datasets

In [30]:
# Load the dataset
ds = load_dataset("Malikeh1375/medical-question-answering-datasets", "chatdoctor_icliniq")
data = ds['train']  # Using the 'train' split

# Debug: Inspect dataset structure
print("Dataset type:", type(data))
print("First item:", data[0])
print("Column names:", data.column_names)

# Print rows 0 to 9 of the dataset
print("\nRows 0 to 9 of the chatdoctor_icliniq dataset:")
try:
    for i, item in enumerate(data.select(range(10))):
        if not isinstance(item, dict):
            print(f"Row {i}: Error - Item is not a dictionary, got {type(item)}: {item}")
            continue
        input_truncated = item['input'][:100] + ("..." if len(item['input']) > 100 else "")
        output_truncated = item['output'][:100] + ("..." if len(item['output']) > 100 else "")
        print(f"\nRow {i}:")
        print(f"Instruction: {item['instruction']}")
        print(f"Input: {input_truncated}")
        print(f"Output: {output_truncated}")
except KeyError as e:
    print(f"Error: Key not found in dataset item - {e}")
except TypeError as e:
    print(f"Error: Type mismatch - {e}")

Dataset type: <class 'datasets.arrow_dataset.Dataset'>
First item: {'instruction': 'Answer this question truthfully', 'input': 'Hello doctor,I had mumps five months ago and after that, I started to have an infection in my left testes. It was swollen and now it has shrunk to almost half the size of the other one. As I am sexually active, I feel a pain in each of the vas deferens after sex. If I do not have sex for days, they become sensitive. I was treated with Ceftum 500 mg, the first time I had an infection. Now my question is, is there any chance that the infection is still in my body? And, do I need to get examined for it? For the time being, please suggest some precautionary antibiotics for my relief.', 'output': 'Hello, Welcome to Chat Doctor forum. I can understand your concern. You had mumps and this is a viral infection known to cause an inflammation of the testis in some cases. Take care. For more information consult a sexologist online'}
Column names: ['instruction', 'input',

Define the Matching Function

In [32]:
def find_best_match(user_question, dataset):
    best_score = 0
    best_answer = "Sorry, I couldn't find a relevant answer. Please try rephrasing your question."
    best_question = None

    # Iterate through the dataset with error handling
    try:
        for item in dataset:
            if not isinstance(item, dict):
                continue  # Skip non-dictionary items
            question = item['input']
            score = fuzz.token_sort_ratio(user_question.lower(), question.lower())
            if score > best_score and score > 50:  # Threshold to avoid irrelevant matches
                best_score = score
                best_answer = item['output']
                best_question = question
    except TypeError as e:
        print(f"Error in matching: {e}")
        return best_answer, best_question, best_score

    return best_answer, best_question, best_score

Define the Chatbot Interface

In [33]:
def chatbot():
    print("Welcome to MediBot: ChatDoctor-iCliniq QA! Type 'exit', 'thank you', 'bye', 'quit', 'done', or 'thanks' to quit.")
    exit_words = ['exit', 'thank you', 'bye', 'quit', 'done', 'thanks']
    farewell_messages = [
        "Take care!",
        "See you later!",
        "Stay healthy!",
        "Farewell!",
        "Have a great day!",
        "Until next time!"
    ]
    while True:
        user_input = input("\nYour question: ").strip()
        if user_input.lower() in exit_words:
            print(random.choice(farewell_messages))
            break

        # Find the best matching answer
        answer, matched_question, score = find_best_match(user_input, data)

        # Display the response with score
        print(f"\nMatch Score: {score}")
        if score > 50:
            print(f"Matched Question: {matched_question}")
            print(f"Answer: {answer}")
        else:
            print(f"{answer}")

Run the Chatbot

In [34]:
if __name__ == "__main__":
    try:
        chatbot()
    except KeyboardInterrupt:
        print("\nInterrupted by user. Take care!")
        sys.exit(0)

Welcome to MediBot: ChatDoctor-iCliniq QA! Type 'exit', 'thank you', 'bye', 'quit', 'done', or 'thanks' to quit.

Your question: Hello doctor i got one side of my wisdom tooth removed

Match Score: 71
Matched Question: Hello doctor,I have wisdom tooth. Is it necessary to remove it?
Answer: Hi, Welcome to Chat Doctor forum. I have gone through your question and I can understand your concern. If it is giving you any sort of problem or giving you discomfort, you need to remove it immediately. There are chances of cyst formation in some cases too, if the impaction is not treated. You can delay it but my opinion is to get it removed.

Your question: thanks
Farewell!
